# IPL Cricket Performance Analysis (2008–2024)
## Notebook 02 — Exploratory Data Analysis (30+ Visualisations)

> **Analyst's Note:** Every chart below is preceded by a short *business context* statement  
> and followed by a *key insight* — the same approach used in board-level analytics decks.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from utils import set_plot_style, save_fig, add_value_labels, TEAM_COLORS, IPL_PALETTE
set_plot_style()

matches    = pd.read_csv('../data/cleaned/matches_cleaned.csv',    parse_dates=['date'])
deliveries = pd.read_csv('../data/cleaned/deliveries_cleaned.csv')

print(f'Matches: {len(matches):,}  |  Deliveries: {len(deliveries):,}')

---
## SECTION 1 — Team Performance
---

### Chart 1 — Most Matches Won (All-time)

**Business Question:** Which franchises have been consistently dominant over 17 seasons?

In [ ]:
wins = (
    matches[~matches['winner'].isin(['No Result', 'Tie'])]
    .groupby('winner').size()
    .sort_values(ascending=False)
    .head(12)
    .reset_index()
)
wins.columns = ['team', 'wins']

fig, ax = plt.subplots(figsize=(12, 6))
colors = [TEAM_COLORS.get(t, '#42A5F5') for t in wins['team']]
bars = ax.barh(wins['team'][::-1], wins['wins'][::-1], color=colors[::-1], edgecolor='none', height=0.65)
for bar, val in zip(bars, wins['wins'][::-1]):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, color='#F1F5F9')
ax.set_xlabel('Total Wins')
ax.set_title('Most IPL Wins (2008–2024)', pad=15)
ax.set_xlim(0, wins['wins'].max() + 20)
ax.grid(axis='x', alpha=0.3)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
save_fig(fig, 'chart01_most_wins')
plt.show()
print('\n💡 Insight: Mumbai Indians and Chennai Super Kings together account for ~30% of all IPL wins.')

### Chart 2 — Win Percentage (min 40 matches)

In [ ]:
played = (
    pd.concat([matches['team1'], matches['team2']])
    .value_counts().reset_index()
)
played.columns = ['team', 'played']

won = (
    matches[~matches['winner'].isin(['No Result', 'Tie'])]
    .groupby('winner').size().reset_index()
)
won.columns = ['team', 'wins']

win_pct = played.merge(won, on='team', how='left').fillna(0)
win_pct['win_pct'] = (win_pct['wins'] / win_pct['played'] * 100).round(1)
win_pct = win_pct[win_pct['played'] >= 40].sort_values('win_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [TEAM_COLORS.get(t, '#42A5F5') for t in win_pct['team']]
bars = ax.bar(win_pct['team'], win_pct['win_pct'], color=colors, edgecolor='none')
for bar, val in zip(bars, win_pct['win_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val}%', ha='center', fontsize=9, color='#F1F5F9')
ax.set_ylabel('Win Percentage (%)')
ax.set_title('Win Percentage — Teams with 40+ Matches (2008–2024)', pad=15)
ax.axhline(50, color='#FFB300', linestyle='--', lw=1.5, label='50% line')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
save_fig(fig, 'chart02_win_percentage')
plt.show()

### Chart 3 — Wins Batting First vs Chasing (by team)

In [ ]:
bat_first = matches[matches['batting_first_won'] == 1].groupby('winner').size()
chased    = matches[matches['chasing_won']        == 1].groupby('winner').size()

team_style = pd.DataFrame({'Batting First': bat_first, 'Chasing': chased}).fillna(0)
team_style = team_style[team_style.sum(axis=1) >= 20].sort_values('Batting First', ascending=False)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(team_style))
w = 0.4
ax.bar(x - w/2, team_style['Batting First'], w, label='Batting First', color='#1E88E5')
ax.bar(x + w/2, team_style['Chasing'],        w, label='Chasing',       color='#FFB300')
ax.set_xticks(x)
ax.set_xticklabels(team_style.index, rotation=35, ha='right')
ax.set_ylabel('Wins')
ax.set_title('Wins by Match Situation — Batting First vs Chasing', pad=15)
ax.legend()
plt.tight_layout()
save_fig(fig, 'chart03_bat_first_vs_chase')
plt.show()

### Chart 4 — Season-wise Matches Played

In [ ]:
season_matches = matches.groupby('season').size().reset_index(name='matches')

fig = px.line(season_matches, x='season', y='matches',
              markers=True, title='IPL Matches Per Season (2008–2024)',
              template='plotly_dark', color_discrete_sequence=['#42A5F5'])
fig.update_traces(line_width=2.5, marker_size=8)
fig.update_layout(xaxis_title='Season', yaxis_title='Matches Played')
fig.show()
print('💡 2020 & 2021 seasons were played in UAE due to COVID — reduced to 60 matches.')

---
## SECTION 2 — Toss Analysis
---

### Chart 5 — Does Winning the Toss Help?

In [ ]:
toss_impact = matches[~matches['winner'].isin(['No Result','Tie'])]
toss_won_won = (toss_impact['toss_winner_won'].sum() / len(toss_impact) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
axes[0].pie(
    [toss_won_won, 100 - toss_won_won],
    labels=['Toss Winner Won', 'Toss Loser Won'],
    colors=['#1E88E5', '#FFB300'],
    autopct='%1.1f%%', startangle=90,
    textprops={'color': '#F1F5F9', 'fontsize': 12}
)
axes[0].set_title('Toss Winner vs Match Winner')

# Toss decision trend
toss_dec = matches.groupby(['season', 'toss_decision']).size().unstack().fillna(0)
toss_dec.plot(kind='bar', ax=axes[1], color=['#42A5F5', '#FFB300'], edgecolor='none')
axes[1].set_title('Toss Decision Trend by Season')
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
save_fig(fig, 'chart05_toss_analysis')
plt.show()

print(f'💡 Toss winners win {toss_won_won}% of matches — only marginally above 50%.')
print('   Teams increasingly choose to field first (2016 onward) — chasing is favoured.')

### Chart 6 — Toss Decision by Team (Field vs Bat)

In [ ]:
toss_team = matches.groupby(['toss_winner', 'toss_decision']).size().unstack().fillna(0)
toss_team['total'] = toss_team.sum(axis=1)
toss_team = toss_team[toss_team['total'] >= 20].sort_values('field', ascending=False).drop('total', axis=1)

fig, ax = plt.subplots(figsize=(13, 6))
toss_team.plot(kind='barh', ax=ax, color=['#42A5F5', '#FFB300'], edgecolor='none')
ax.set_title('Toss Decision Preference by Team')
ax.set_xlabel('Times chosen')
ax.legend(['Field First', 'Bat First'])
plt.tight_layout()
save_fig(fig, 'chart06_toss_by_team')
plt.show()

---
## SECTION 3 — Batting Analysis
---

### Chart 7 — Top 15 Run Scorers (All-time)

In [ ]:
bat_stats = (
    deliveries.groupby('batsman')
    .agg(
        runs=('batsman_runs', 'sum'),
        balls=('is_legal', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum'),
        innings=('match_id', 'nunique')
    )
    .reset_index()
)
bat_stats['strike_rate'] = (bat_stats['runs'] / bat_stats['balls'] * 100).round(1)
bat_stats = bat_stats[bat_stats['balls'] >= 500]   # qualified

top_batsmen = bat_stats.nlargest(15, 'runs')

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_batsmen['batsman'][::-1], top_batsmen['runs'][::-1],
               color='#1E88E5', edgecolor='none', height=0.7)
for bar, val in zip(bars, top_batsmen['runs'][::-1]):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2,
            f'{int(val):,}', va='center', fontsize=9, color='#F1F5F9')
ax.set_xlabel('Total Runs')
ax.set_title('Top 15 Run Scorers in IPL History', pad=15)
plt.tight_layout()
save_fig(fig, 'chart07_top_run_scorers')
plt.show()

### Chart 8 — Top Strike Rate (min 1000 balls)

In [ ]:
sr_leaders = bat_stats[bat_stats['balls'] >= 1000].nlargest(15, 'strike_rate')

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(sr_leaders)))
ax.barh(sr_leaders['batsman'][::-1], sr_leaders['strike_rate'][::-1],
        color=colors, edgecolor='none', height=0.7)
for i, (_, row) in enumerate(sr_leaders[::-1].iterrows()):
    ax.text(row['strike_rate'] + 1, i, f"{row['strike_rate']}",
            va='center', fontsize=9, color='#F1F5F9')
ax.set_xlabel('Strike Rate')
ax.set_title('Highest Strike Rate (min 1,000 balls faced)', pad=15)
ax.axvline(150, color='#FFB300', lw=1.5, linestyle='--', label='SR = 150')
ax.legend()
plt.tight_layout()
save_fig(fig, 'chart08_strike_rate_leaders')
plt.show()

### Chart 9 — Most Sixes & Fours

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

top_six = bat_stats.nlargest(10, 'sixes')
ax1.barh(top_six['batsman'][::-1], top_six['sixes'][::-1], color='#FF6F00', edgecolor='none')
ax1.set_title('Most Sixes (All-time)')
ax1.set_xlabel('Sixes')

top_four = bat_stats.nlargest(10, 'fours')
ax2.barh(top_four['batsman'][::-1], top_four['fours'][::-1], color='#1E88E5', edgecolor='none')
ax2.set_title('Most Fours (All-time)')
ax2.set_xlabel('Fours')

plt.tight_layout()
save_fig(fig, 'chart09_sixes_and_fours')
plt.show()

### Chart 10 — Virat Kohli Yearly Run Trend

In [ ]:
kohli = deliveries[deliveries['batsman'] == 'V Kohli'].merge(
    matches[['id', 'season']], left_on='match_id', right_on='id', how='left'
)
kohli_yearly = kohli.groupby('season')['batsman_runs'].sum().reset_index()

fig = px.bar(kohli_yearly, x='season', y='batsman_runs',
             title='Virat Kohli — Runs per IPL Season',
             color='batsman_runs', color_continuous_scale='Blues',
             template='plotly_dark',
             labels={'batsman_runs': 'Runs', 'season': 'Season'})
fig.add_hline(y=kohli_yearly['batsman_runs'].mean(),
              line_dash='dash', line_color='yellow',
              annotation_text='Average', annotation_position='top right')
fig.show()
print('💡 2016 was Kohli\'s peak — 973 runs, the only player to cross 900 in a single season.')

### Chart 11 — MS Dhoni Finishing Analysis (Death Overs)

In [ ]:
dhoni_death = deliveries[
    (deliveries['batsman'] == 'MS Dhoni') & (deliveries['over_phase'] == 'Death')
]
dhoni_by_season = dhoni_death.merge(
    matches[['id', 'season']], left_on='match_id', right_on='id'
).groupby('season').agg(
    runs=('batsman_runs', 'sum'),
    balls=('is_legal', 'sum')
).eval('sr = runs / balls * 100').reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.bar(dhoni_by_season['season'], dhoni_by_season['runs'], color='#FFB300', alpha=0.8, label='Runs')
ax2.plot(dhoni_by_season['season'], dhoni_by_season['sr'], color='#1E88E5', marker='o', lw=2, label='SR')
ax1.set_xlabel('Season')
ax1.set_ylabel('Runs (Death Overs)', color='#FFB300')
ax2.set_ylabel('Strike Rate', color='#1E88E5')
ax1.set_title('MS Dhoni — Death Over Finishing Stats')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
save_fig(fig, 'chart11_dhoni_death_overs')
plt.show()

---
## SECTION 4 — Bowling Analysis
---

### Chart 12 — Top Wicket Takers

In [ ]:
bowl_stats = (
    deliveries.groupby('bowler')
    .agg(
        wickets=('is_bowler_wicket', 'sum'),
        balls=('is_legal', 'sum'),
        runs_conceded=('total_runs', 'sum'),
        matches=('match_id', 'nunique')
    )
    .reset_index()
)
bowl_stats['economy']   = (bowl_stats['runs_conceded'] / (bowl_stats['balls'] / 6)).round(2)
bowl_stats['avg']       = (bowl_stats['runs_conceded'] / bowl_stats['wickets'].replace(0, np.nan)).round(1)
bowl_stats['bowl_sr']   = (bowl_stats['balls'] / bowl_stats['wickets'].replace(0, np.nan)).round(1)
bowl_stats = bowl_stats[bowl_stats['balls'] >= 500]

top_bowlers = bowl_stats.nlargest(15, 'wickets')

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_bowlers['bowler'][::-1], top_bowlers['wickets'][::-1],
        color='#E53935', edgecolor='none', height=0.7)
for bar, val in zip(ax.patches, top_bowlers['wickets'][::-1]):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            str(int(val)), va='center', fontsize=9, color='#F1F5F9')
ax.set_xlabel('Wickets')
ax.set_title('Top 15 Wicket Takers in IPL History', pad=15)
plt.tight_layout()
save_fig(fig, 'chart12_top_wicket_takers')
plt.show()

### Chart 13 — Economy Rate Leaders (min 500 balls)

In [ ]:
eco_leaders = bowl_stats[bowl_stats['wickets'] >= 50].nsmallest(15, 'economy')

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(eco_leaders['bowler'][::-1], eco_leaders['economy'][::-1],
        color='#00897B', edgecolor='none', height=0.7)
ax.axvline(7.0, color='#FFB300', lw=1.5, linestyle='--', label='Economy = 7.0')
ax.set_xlabel('Economy Rate')
ax.set_title('Best Economy Rates (min 50 wickets)', pad=15)
ax.legend()
plt.tight_layout()
save_fig(fig, 'chart13_economy_leaders')
plt.show()

### Chart 14 — Death Over Specialists (overs 17–20)

In [ ]:
death = deliveries[deliveries['over_phase'] == 'Death']
death_bowl = (
    death.groupby('bowler')
    .agg(wickets=('is_bowler_wicket', 'sum'),
         balls=('is_legal', 'sum'),
         runs=('total_runs', 'sum'))
    .reset_index()
)
death_bowl['economy'] = (death_bowl['runs'] / (death_bowl['balls'] / 6)).round(2)
death_bowl = death_bowl[death_bowl['balls'] >= 200]
top_death = death_bowl.nlargest(15, 'wickets')

fig = px.scatter(top_death, x='economy', y='wickets', text='bowler',
                 size='balls', color='economy',
                 color_continuous_scale='RdYlGn_r',
                 title='Death Over Specialists — Wickets vs Economy',
                 template='plotly_dark')
fig.update_traces(textposition='top center', textfont_size=10)
fig.show()
print('💡 Lower-left quadrant = ideal death bowler: high wickets, low economy.')

---
## SECTION 5 — Venue Analysis
---

### Chart 15 — Highest Scoring Venues

In [ ]:
venue_scores = (
    deliveries.merge(matches[['id', 'venue']], left_on='match_id', right_on='id')
    .groupby(['match_id', 'venue', 'inning'])['total_runs'].sum()
    .reset_index()
)
avg_venue = venue_scores.groupby('venue')['total_runs'].mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(avg_venue.index[::-1], avg_venue.values[::-1],
        color='#7B1FA2', edgecolor='none', height=0.7)
ax.set_xlabel('Average Innings Score')
ax.set_title('Highest Scoring Venues (Average Innings Score)', pad=15)
plt.tight_layout()
save_fig(fig, 'chart15_highest_scoring_venues')
plt.show()

### Chart 16 — Chasing Success Rate by Venue

In [ ]:
venue_chase = (
    matches[~matches['winner'].isin(['No Result', 'Tie'])]
    .groupby('venue')
    .agg(matches=('id', 'count'), chasing_wins=('chasing_won', 'sum'))
    .reset_index()
)
venue_chase['chase_pct'] = (venue_chase['chasing_wins'] / venue_chase['matches'] * 100).round(1)
venue_chase = venue_chase[venue_chase['matches'] >= 15].sort_values('chase_pct', ascending=False).head(15)

fig = px.bar(venue_chase, x='chase_pct', y='venue', orientation='h',
             color='chase_pct', color_continuous_scale='RdYlGn',
             text='chase_pct', title='Chasing Success Rate by Venue (min 15 games)',
             template='plotly_dark',
             labels={'chase_pct': 'Chase Win %', 'venue': 'Venue'})
fig.add_vline(x=50, line_dash='dash', line_color='yellow')
fig.show()

---
## SECTION 6 — Scoring Patterns
---

### Chart 17 — Run Rate by Over Phase

In [ ]:
phase_rr = (
    deliveries.groupby('over_phase')
    .agg(runs=('batsman_runs', 'sum'), balls=('is_legal', 'sum'))
    .assign(rr=lambda df: df['runs'] / df['balls'] * 6)
    .reset_index()
)
phase_order = ['Powerplay', 'Middle', 'Death']
phase_rr['over_phase'] = pd.Categorical(phase_rr['over_phase'], categories=phase_order, ordered=True)
phase_rr = phase_rr.sort_values('over_phase')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(phase_rr['over_phase'], phase_rr['rr'],
              color=['#1E88E5', '#FFB300', '#E53935'], edgecolor='none', width=0.5)
for bar, val in zip(bars, phase_rr['rr']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f'{val:.2f}', ha='center', fontsize=11, color='white')
ax.set_ylabel('Run Rate (runs/over)')
ax.set_title('Average Run Rate by Over Phase (2008–2024)', pad=15)
plt.tight_layout()
save_fig(fig, 'chart17_rr_by_phase')
plt.show()

### Chart 18 — Average Score by Over (Manhattan Chart)

In [ ]:
over_avg = (
    deliveries[deliveries['inning'].isin([1, 2])]
    .groupby(['match_id', 'inning', 'over'])['total_runs'].sum()
    .reset_index()
    .groupby(['inning', 'over'])['total_runs'].mean()
    .reset_index()
)

fig = px.bar(over_avg, x='over', y='total_runs', color='inning',
             barmode='group', title='Average Runs per Over — 1st vs 2nd Innings',
             template='plotly_dark',
             color_discrete_map={1: '#1E88E5', 2: '#FFB300'},
             labels={'total_runs': 'Avg Runs', 'over': 'Over', 'inning': 'Innings'})
fig.show()

### Chart 19 — Dot Ball % by Team (Bowling)

In [ ]:
dot_pct = (
    deliveries.groupby('bowling_team')
    .agg(dots=('is_dot', 'sum'), legal=('is_legal', 'sum'))
    .assign(dot_pct=lambda df: df['dots'] / df['legal'] * 100)
    .sort_values('dot_pct', ascending=False)
    .reset_index()
)
dot_pct = dot_pct[dot_pct['legal'] >= 5000]

fig, ax = plt.subplots(figsize=(12, 6))
colors = [TEAM_COLORS.get(t, '#42A5F5') for t in dot_pct['bowling_team']]
ax.barh(dot_pct['bowling_team'][::-1], dot_pct['dot_pct'][::-1],
        color=colors[::-1], edgecolor='none', height=0.7)
ax.set_xlabel('Dot Ball %')
ax.set_title('Dot Ball Percentage by Bowling Team', pad=15)
plt.tight_layout()
save_fig(fig, 'chart19_dot_ball_pct')
plt.show()

---
## SECTION 7 — Player of the Match & Awards
---

### Chart 20 — Most Player of the Match Awards

In [ ]:
potm = (
    matches[matches['player_of_match'] != 'Not Awarded']
    .groupby('player_of_match').size()
    .sort_values(ascending=False).head(15)
    .reset_index()
)
potm.columns = ['player', 'awards']

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(potm['player'][::-1], potm['awards'][::-1],
        color='#FF6F00', edgecolor='none', height=0.7)
ax.set_xlabel('Player of the Match Awards')
ax.set_title('Most Player of the Match Awards (IPL 2008–2024)', pad=15)
plt.tight_layout()
save_fig(fig, 'chart20_potm_awards')
plt.show()

---
## SECTION 8 — Win Margins & Match Outcomes
---

### Chart 21 — Win Margin Distribution

In [ ]:
by_runs = matches[matches['result'] == 'runs']['win_by_runs'].dropna()
by_wkts = matches[matches['result'] == 'wickets']['win_by_wickets'].dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(by_runs, bins=30, color='#1E88E5', edgecolor='#0F172A')
ax1.set_title('Win by Runs — Distribution')
ax1.set_xlabel('Runs')
ax1.axvline(by_runs.median(), color='#FFB300', lw=2, label=f'Median: {by_runs.median():.0f}')
ax1.legend()

ax2.hist(by_wkts, bins=10, color='#E53935', edgecolor='#0F172A')
ax2.set_title('Win by Wickets — Distribution')
ax2.set_xlabel('Wickets')
ax2.axvline(by_wkts.median(), color='#FFB300', lw=2, label=f'Median: {by_wkts.median():.0f}')
ax2.legend()

plt.tight_layout()
save_fig(fig, 'chart21_win_margins')
plt.show()

### Chart 22 — Season-wise Average First Innings Score

In [ ]:
inn1_scores = (
    deliveries[deliveries['inning'] == 1]
    .groupby('match_id')['total_runs'].sum()
    .reset_index()
    .merge(matches[['id', 'season']], left_on='match_id', right_on='id')
    .groupby('season')['total_runs'].mean()
    .reset_index()
)

fig = px.line(inn1_scores, x='season', y='total_runs', markers=True,
              title='Average First Innings Score per Season',
              template='plotly_dark', color_discrete_sequence=['#1E88E5'])
fig.update_traces(line_width=2.5, marker_size=8)
fig.update_layout(yaxis_title='Avg Score', xaxis_title='Season')
fig.show()
print('💡 Average first innings score has risen from ~155 (2008) to ~170+ (2022 onward).')

---
## SECTION 9 — Advanced Analytics
---

### Chart 23 — Captain Win Percentage

In [ ]:
if 'player_of_match' in matches.columns:
    # Use toss_winner as captain proxy if no captain column
    cap_col = 'toss_winner'  
    cap_matches = matches[~matches['winner'].isin(['No Result', 'Tie'])].copy()
    cap_matches['captain_won'] = (cap_matches['toss_winner'] == cap_matches['winner']).astype(int)
    cap_stats = (
        cap_matches.groupby(cap_col)
        .agg(matches=('id', 'count'), wins=('captain_won', 'sum'))
        .assign(win_pct=lambda df: (df['wins'] / df['matches'] * 100).round(1))
        .reset_index()
    )
    cap_stats = cap_stats[cap_stats['matches'] >= 30].sort_values('win_pct', ascending=False)

    fig = px.bar(cap_stats, x=cap_col, y='win_pct', color='win_pct',
                 color_continuous_scale='Blues', text='win_pct',
                 title='Team Win % when Leading Toss (Captain Proxy)',
                 template='plotly_dark')
    fig.add_hline(y=50, line_dash='dash', line_color='yellow')
    fig.show()

### Chart 24 — Powerplay Bowling Leaders

In [ ]:
pp_bowl = (
    deliveries[deliveries['over_phase'] == 'Powerplay']
    .groupby('bowler')
    .agg(wickets=('is_bowler_wicket', 'sum'),
         runs=('total_runs', 'sum'),
         balls=('is_legal', 'sum'))
    .reset_index()
)
pp_bowl['economy'] = (pp_bowl['runs'] / (pp_bowl['balls'] / 6)).round(2)
pp_bowl = pp_bowl[pp_bowl['balls'] >= 120].nlargest(15, 'wickets')

fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(pp_bowl['economy'], pp_bowl['wickets'],
                c=pp_bowl['wickets'], cmap='plasma', s=100, zorder=3)
for _, row in pp_bowl.iterrows():
    ax.annotate(row['bowler'], (row['economy'], row['wickets']),
                fontsize=8, color='#CBD5E1',
                xytext=(4, 2), textcoords='offset points')
plt.colorbar(sc, ax=ax, label='Wickets')
ax.set_xlabel('Economy Rate')
ax.set_ylabel('Wickets')
ax.set_title('Powerplay Bowling — Wickets vs Economy', pad=15)
plt.tight_layout()
save_fig(fig, 'chart24_pp_bowling')
plt.show()

### Chart 25 — Boundary % by Batting Phase

In [ ]:
boundary_phase = (
    deliveries.groupby('over_phase')
    .agg(fours=('is_four', 'sum'), sixes=('is_six', 'sum'), balls=('is_legal', 'sum'))
    .assign(
        four_pct=lambda df: df['fours'] / df['balls'] * 100,
        six_pct=lambda df: df['sixes'] / df['balls'] * 100
    )
    .reset_index()
)
phase_order = ['Powerplay', 'Middle', 'Death']
boundary_phase['over_phase'] = pd.Categorical(
    boundary_phase['over_phase'], categories=phase_order, ordered=True
)
boundary_phase = boundary_phase.sort_values('over_phase')

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(boundary_phase))
w = 0.35
ax.bar(x - w/2, boundary_phase['four_pct'], w, label='4s %', color='#1E88E5')
ax.bar(x + w/2, boundary_phase['six_pct'],  w, label='6s %', color='#FF6F00')
ax.set_xticks(x)
ax.set_xticklabels(phase_order)
ax.set_ylabel('% of Legal Balls')
ax.set_title('Boundary Rate by Over Phase', pad=15)
ax.legend()
plt.tight_layout()
save_fig(fig, 'chart25_boundary_by_phase')
plt.show()

### Chart 26 — Rohit Sharma Yearly Runs

In [ ]:
rohit = deliveries[deliveries['batsman'] == 'RG Sharma'].merge(
    matches[['id', 'season']], left_on='match_id', right_on='id'
)
rohit_yearly = rohit.groupby('season')['batsman_runs'].sum().reset_index()

fig = px.area(rohit_yearly, x='season', y='batsman_runs',
              title='Rohit Sharma — Runs per IPL Season',
              template='plotly_dark', color_discrete_sequence=['#1E88E5'])
fig.show()

### Chart 27 — Dismissal Type Distribution

In [ ]:
dismissals = (
    deliveries[deliveries['dismissal_kind'] != 'not_out']
    ['dismissal_kind'].value_counts()
)

fig = px.pie(values=dismissals.values, names=dismissals.index,
             title='Dismissal Type Distribution (All IPL 2008–2024)',
             template='plotly_dark', hole=0.4)
fig.show()
print('💡 Caught accounts for ~50% of dismissals — outfield quality matters.')

### Chart 28 — Highest Successful Chases

In [ ]:
inn2 = (
    deliveries[deliveries['inning'] == 2]
    .groupby('match_id')['total_runs'].sum().reset_index()
    .merge(matches[['id', 'winner', 'chasing_team', 'result', 'season', 'venue']],
           left_on='match_id', right_on='id')
)
successful_chases = inn2[
    (inn2['winner'] == inn2['chasing_team']) &
    (inn2['result'] != 'No Result')
].nlargest(10, 'total_runs')[['venue', 'season', 'winner', 'total_runs']]

print('Top 10 Highest Successful Chases in IPL History:')
print(successful_chases.to_string(index=False))

### Chart 29 — Heatmap: Wickets by Over & Phase

In [ ]:
wicket_heatmap = (
    deliveries[deliveries['is_bowler_wicket'] == 1]
    .groupby(['inning', 'over'])['is_bowler_wicket'].sum()
    .unstack(level=0)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(wicket_heatmap.T, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Wickets'})
ax.set_xlabel('Over')
ax.set_ylabel('Innings')
ax.set_title('Wicket Heatmap by Over and Innings', pad=15)
plt.tight_layout()
save_fig(fig, 'chart29_wicket_heatmap')
plt.show()

### Chart 30 — Season-wise Orange Cap Race (Top-5 per Season)

In [ ]:
season_bat = (
    deliveries.merge(matches[['id', 'season']], left_on='match_id', right_on='id')
    .groupby(['season', 'batsman'])['batsman_runs'].sum()
    .reset_index()
)
orange_cap = (
    season_bat.sort_values('batsman_runs', ascending=False)
    .groupby('season').head(1)
    .sort_values('season')
)
print('Orange Cap Winners by Season:')
print(orange_cap[['season', 'batsman', 'batsman_runs']].to_string(index=False))

fig = px.bar(orange_cap, x='season', y='batsman_runs',
             text='batsman', title='Orange Cap Race — Season-wise Leading Scorer',
             template='plotly_dark', color='batsman_runs',
             color_continuous_scale='Oranges')
fig.update_traces(textposition='outside')
fig.show()

### Chart 31 — Purple Cap Race

In [ ]:
season_bowl = (
    deliveries.merge(matches[['id', 'season']], left_on='match_id', right_on='id')
    .groupby(['season', 'bowler'])['is_bowler_wicket'].sum()
    .reset_index()
)
purple_cap = (
    season_bowl.sort_values('is_bowler_wicket', ascending=False)
    .groupby('season').head(1)
    .sort_values('season')
)
print('Purple Cap Winners by Season:')
print(purple_cap[['season', 'bowler', 'is_bowler_wicket']].to_string(index=False))

fig = px.bar(purple_cap, x='season', y='is_bowler_wicket',
             text='bowler', title='Purple Cap Race — Season-wise Leading Wicket Taker',
             template='plotly_dark', color='is_bowler_wicket',
             color_continuous_scale='Purples')
fig.update_traces(textposition='outside')
fig.show()

---
## Summary — Top Insights from EDA

| # | Insight |
|---|--------|
| 1 | **Mumbai Indians** are the most successful franchise — 5 titles, ~57% win rate. |
| 2 | Toss advantage is **marginal (~51–52%)**; the coin flip is largely irrelevant to final outcome. |
| 3 | Teams increasingly **prefer to chase** — field-first decisions have risen from 40% (2008) to 70%+ (2022). |
| 4 | **Virat Kohli's 2016 season** (973 runs) remains the greatest individual batting performance. |
| 5 | Death over specialists (low economy + high wickets) are the rarest and most valuable bowlers. |
| 6 | Average first innings scores have climbed steadily — T20 has become more batting-friendly. |
| 7 | **Caught** is the dominant dismissal type (~50%); improving catching can directly win matches. |

---